In [1]:
import pandas as pd
import numpy as np

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

X_valid = pd.read_csv("../data/processed/X_valid.csv")
y_valid = pd.read_csv("../data/processed/y_valid.csv").squeeze()

In [2]:
print(y_train.value_counts())

print(
    y_train.value_counts(normalize=True) * 100
)

Churn
0    3104
1    1121
Name: count, dtype: int64
Churn
0    73.467456
1    26.532544
Name: proportion, dtype: float64


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import cross_validate, StratifiedKFold

In [4]:
import joblib

In [5]:
preprocessor = joblib.load(
    "../models/preprocessor.joblib"
)

models = {
    "Baseline": LogisticRegression(
        max_iter=2000
    ),

    "Class Weight": LogisticRegression(
        class_weight="balanced",
        max_iter=2000
    ),

    "SMOTE": ImbPipeline(steps=[
        ("smote", SMOTE(
            random_state=42,
            k_neighbors=5
        )),
        ("model", LogisticRegression(
            max_iter=2000
        ))
    ])
}

In [6]:
def build_pipeline(model):
    return ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("sampling", "passthrough"),
        ("classifier", model)
    ])

In [7]:
import joblib

preprocessor = joblib.load("../models/preprocessor.joblib")

In [8]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

imbalance_models = {
    "Baseline": ImbPipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ]),

    "Class Weight": ImbPipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=2000
        ))
    ]),

    "SMOTE": ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", LogisticRegression(max_iter=2000))
    ])
}

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, model in imbalance_models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "f1": "f1",
            "recall": "recall",
            "average_precision": "average_precision"
        },
        n_jobs=-1
    )

    results.append({
        "Method": name,
        "F1": scores["test_f1"].mean(),
        "Recall": scores["test_recall"].mean(),
        "PR_AUC": scores["test_average_precision"].mean()
    })

imbalance_results = pd.DataFrame(results)

imbalance_results.sort_values(
    by="PR_AUC",
    ascending=False
)

,Method,F1,Recall,PR_AUC
0,Baseline,0.595558,0.546857,0.662888
1,Class Weight,0.619927,0.793087,0.658649
2,SMOTE,0.630866,0.792179,0.656027
